# 9. HyDE - Hypothetical Document Embedding

Search with the embedding of a *hypothetical answer* instead of the question.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Embedding models place questions and answers in slightly different regions of vector space: a question like *'why does reranking help?'* looks different from the passage that answers it. **HyDE** asks the LLM to write a plausible (possibly wrong) answer first and searches with *that* text, because a fake answer looks more like real documents than a question does.

## How it works

1. LLM writes a hypothetical answer passage
2. Embed the hypothetical passage (not the question)
3. Search the index with that vector
4. Generate the real answer from the retrieved real documents
5. The hypothetical text is thrown away

```text
 question --> LLM --> hypothetical answer --> embed --> search real chunks --> LLM --> answer
              (may contain mistakes; only used as a search key)
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 10:39:13.567000 7344 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


Questions and answers live in different parts of embedding space. An answer-shaped query lands closer to answer-shaped chunks.

In [2]:
def hyde_search(query, k=3):
    fake = llm(f"Write a short passage (3-4 sentences) that answers the question. It may contain inaccuracies.\n\nQuestion: {query}")
    return fake, store.search(fake, k)

query = "Why does adding a second look at the top results make retrieval more accurate?"
fake, hyde_hits = hyde_search(query)
print("Hypothetical document:\n", fake)

Hypothetical document:
 Adding a second look at the top of a document or webpage is believed to improve retrieval accuracy because it allows users to quickly scan and identify the most relevant information before diving deeper into the content. This initial glance, often referred to as "top-down processing," enables users to make a preliminary judgment about what they're looking for, thereby reducing the time spent searching for specific information. By doing so, users can focus their search efforts on more promising areas of the document, leading to faster and more accurate retrieval.


In [3]:
print("--- HyDE"); show(hyde_hits)
print("--- direct query"); show(store.search(query, 3))

--- HyDE
0.464  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted score or with Reciprocal Rank Fusion, which adds 1 divided by...
0.427  hunking Strategies  Documents must be split before indexing because embedding a whole book into one vector blurs its meaning. Fixed-size chunking cuts text ever...
0.411  index, optionally reranking and finally generating the answer, with generation usually the slowest step. Semantic caching stores previous answers and returns th...
--- direct query
0.424  t paraphrases and meaning. Hybrid search runs both and merges the rankings, either with a weighted score or with Reciprocal Rank Fusion, which adds 1 divided by...
0.390  index, optionally reranking and finally generating the answer, with generation usually the slowest step. Semantic caching stores previous answers and returns th...
0.330  r and outputs a relevance score, which is far more accurate than comparing two independent vectors. Because cros

In [4]:
print(answer(query, [h["text"] for h in hyde_hits]))

According to the context, adding a second look (reranking) makes retrieval more accurate because it uses a cross-encoder that reads the query and a candidate passage together and outputs a relevance score. This step is said to be "far more precise" than the first-stage retriever, which is described as fast but coarse.


**Watch out:** if the LLM knows nothing about the topic, the hypothetical document can mislead retrieval. Combine with the original query for safety.

## Combine HyDE with the original query
A hallucinated passage can drift. Averaging the two embeddings (or merging both result lists) keeps the user's intent.

In [5]:
import numpy as np
def hyde_plus_query(query, k=3):
    fake = llm(f"Write a short passage (3-4 sentences) that answers the question.\n\nQuestion: {query}")
    v = embed(fake)[0] + embed(query)[0]
    v = v / np.linalg.norm(v)
    scores = store.vecs @ v
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

## Evaluate on the golden set

In [6]:
compare_retrievers({
    "direct query": store.search,
    "HyDE": lambda q, k=3: hyde_search(q, k)[1],
    "HyDE + query": hyde_plus_query,
}, k=3)

,hit@1,hit@3,MRR,ranks
method,,,,
direct query,0.80,1.0,0.89,"[1, 2, 2, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, ..."
HyDE,0.85,1.0,0.93,"[1, 1, 2, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, ..."
HyDE + query,0.75,1.0,0.87,"[1, 2, 2, 1, 3, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, ..."


## When HyDE hurts: the model knows nothing
For private jargon the LLM invents a confident but wrong passage. Look at what it writes for a made-up term.

In [7]:
q = "What does the Zorblax-7 calibration routine do?"
fake, hits = hyde_search(q)
print("hypothetical:", fake[:300], "\n")
show(hits, width=90)

hypothetical: The Zorblax-7 calibration routine is an ancient protocol used by intergalactic engineers to realign the dimensional harmonics of a spaceship's propulsion system. By performing this intricate series of calculations and energy resonance adjustments, the Zorblax-7 calibration routine ensures that the v 

0.182  user or group identifiers. Running models locally with a tool such as Ollama keeps sensiti...
0.181  r and outputs a relevance score, which is far more accurate than comparing two independent...
0.171  cuments and falls back to web search when they are irrelevant. Self-RAG trains the model t...


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| No re-indexing, no training | One extra LLM call per query |
| Helps short/abstract questions | Hallucinated passages can mislead retrieval for unknown topics |
| Works with any retriever | Little gain when queries are already descriptive |

## When to use it

Zero-shot retrieval over domains where the LLM has decent background knowledge but queries are short.

## Experiments to try

- Generate several hypothetical documents and average their embeddings.
- Ask for the answer in the style of your corpus (e.g. 'write like a handbook').
- Measure the added latency.

## Key takeaways

- Search with an answer-shaped query.
- Keep the original query in the mix.
- Beware of unknown domains.

---
**Next:** [10. HyPE - Hypothetical Prompt Embeddings](../10_HyPE_Hypothetical_Prompt_Embeddings/HyPE_Hypothetical_Prompt_Embeddings.ipynb)